# Data Quality

Validation, duplicates, null checks, quarantine and audit metrics.

Synthetic retail case study. Run from the repository root.

In [ ]:
from pathlib import Path
import pandas as pd
from pyspark.sql import SparkSession, functions as F, types as T
spark = SparkSession.builder.master("local[2]").appName("SparkLearning").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
root = Path.cwd()
if not (root / "data" / "retail_orders_messy.csv").exists():
    root = root.parent
orders_path = str(root / "data" / "retail_orders_messy.csv")
customers_path = str(root / "data" / "customers.csv")
assert Path(orders_path).exists(), "Run Jupyter from the repository root, or update orders_path"
orders = spark.read.option("header", True).option("inferSchema", True).csv(orders_path)
customers = spark.read.option("header", True).option("inferSchema", True).csv(customers_path)
print("Raw rows:", orders.count())
orders.printSchema()
orders.show(5, truncate=False)

## Profile missingness

In [ ]:
for name in orders.columns:
    print(name, orders.filter(F.col(name).isNull() | (F.trim(F.col(name).cast("string")) == "")).count())

## Validate and quarantine

In [ ]:
# Normalize types and validate before aggregating
normalized = (orders
    .withColumn("parsed_date", F.to_date("order_date", "yyyy-MM-dd"))
    .withColumn("quantity_num", F.col("quantity").cast("int"))
    .withColumn("unit_price_num", F.col("unit_price").cast("double"))
    .withColumn("region_clean", F.trim(F.col("region"))))
valid_expr = (F.col("parsed_date").isNotNull() & F.col("quantity_num").isNotNull() &
              (F.col("quantity_num") > 0) & F.col("unit_price_num").isNotNull() &
              (F.col("unit_price_num") >= 0) & F.col("region_clean").isNotNull() &
              (F.length("region_clean") > 0))
deduped = normalized.dropDuplicates(["order_id", "order_date", "customer_id", "region", "category", "quantity", "unit_price", "status"])
accepted = deduped.filter(valid_expr)
quarantine = deduped.filter(~valid_expr)
print("Input:", orders.count(), "Deduplicated:", deduped.count(), "Accepted:", accepted.count(), "Quarantine:", quarantine.count())
assert accepted.count() + quarantine.count() == deduped.count()
quarantine.select("order_id", "order_date", "region", "quantity", "unit_price").show()

## Quality score

In [ ]:
print("Acceptance rate:", round(accepted.count() / deduped.count() * 100, 2), "%")

## Reflect

What quality issues did you observe? Which transformations trigger a shuffle? What would you change for production-scale data?

In [ ]:
spark.stop()